# Evaluating RAG with Ragas: Naive vs. Semantic Chunking

This notebook presents a comprehensive evaluation of two different chunking strategies for Retrieval-Augmented Generation (RAG) systems. We will be evaluating with **Ragas** to compare a **naive chunking approach** using RecursiveCharacterTextSplitter against a **semantic chunking strategy** using LangChain's SemanticChunker to determine which method produces better retrieval and generation performance. The evaluation leverages synthetic question-answer pairs generated from domain-specific documents and employs five key metrics: Faithfulness, Response Relevancy, Context Precision, Context Recall, and Answer Correctness. Through this systematic comparison, we aim to demonstrate the impact of chunking strategy on overall RAG system performance and provide insights into optimal document processing techniques for knowledge retrieval applications.


In [1]:
# OpenAI setup
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")


In [2]:
# LangSmith setup for visualization
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass("Enter your LangSmith API key: ")
os.environ["LANGCHAIN_PROJECT"] = "RAG-Evaluation-Ragas"

# Part 1: Synthetic Dataset Generation

For the creation of the knowledge graph, we will be using the **abstracted method** that automatically handles the complexity of manually creating the graph. This approach leverages Ragas's built-in `TestsetGenerator` which internally constructs a knowledge graph from our documents, extracts entities and relationships, and uses this structured representation to generate sophisticated, interconnected questions without requiring us to manually define graph nodes, edges, or traversal logic.

In [3]:
# Data preparation
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader

path = "data/"
loader = DirectoryLoader(path, glob = "*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()



In [4]:
print(len(docs)) # PyMuPDFLoader has splitted the doc into pages by default

64


In [5]:
# Knowledge Graph
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


c:\Users\Inés\AIE2\08_Evaluating_RAG_With_Ragas\.venv\Lib\site-packages\pysbd\segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
c:\Users\Inés\AIE2\08_Evaluating_RAG_With_Ragas\.venv\Lib\site-packages\pysbd\lang\arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
c:\Users\Inés\AIE2\08_Evaluating_RAG_With_Ragas\.venv\Lib\site-packages\pysbd\lang\persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


In [6]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm= generator_llm, embedding_model= generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/39 [00:00<?, ?it/s]

Property 'summary' already exists in node '08c29b'. Skipping!
Property 'summary' already exists in node '8e2827'. Skipping!
Property 'summary' already exists in node '4f1edf'. Skipping!
Property 'summary' already exists in node '7c7093'. Skipping!
Property 'summary' already exists in node '1b92d0'. Skipping!
Property 'summary' already exists in node 'd8a62b'. Skipping!
Property 'summary' already exists in node 'c66e3f'. Skipping!
Property 'summary' already exists in node '4f2b42'. Skipping!
Property 'summary' already exists in node '2e20f1'. Skipping!
Property 'summary' already exists in node 'a6715c'. Skipping!
Property 'summary' already exists in node 'ce099e'. Skipping!
Property 'summary' already exists in node '4e92ef'. Skipping!
Property 'summary' already exists in node 'c6277b'. Skipping!
Property 'summary' already exists in node '791f9d'. Skipping!
Property 'summary' already exists in node '8b2668'. Skipping!
Property 'summary' already exists in node 'eb6e77'. Skipping!
Property

...

Property 'summary_embedding' already exists in node 'eb6e77'. Skipping!
Property 'summary_embedding' already exists in node '791f9d'. Skipping!
Property 'summary_embedding' already exists in node '7c7093'. Skipping!
Property 'summary_embedding' already exists in node '08c29b'. Skipping!
Property 'summary_embedding' already exists in node '4f2b42'. Skipping!
Property 'summary_embedding' already exists in node '4f1edf'. Skipping!
Property 'summary_embedding' already exists in node '8e2827'. Skipping!
Property 'summary_embedding' already exists in node 'c66e3f'. Skipping!
Property 'summary_embedding' already exists in node '4e92ef'. Skipping!
Property 'summary_embedding' already exists in node '8b2668'. Skipping!
Property 'summary_embedding' already exists in node 'ce099e'. Skipping!
Property 'summary_embedding' already exists in node '1b92d0'. Skipping!
Property 'summary_embedding' already exists in node 'a6715c'. Skipping!
Property 'summary_embedding' already exists in node 'd8a62b'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/8 [00:00<?, ?it/s]

In [7]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,how artificial intelligence like chatgpt get s...,[Introduction ChatGPT launched in November 202...,"ChatGPT, which is based on a Large Language Mo...",single_hop_specifc_query_synthesizer
1,"When was ChatGPT launched, and what significan...",[Introduction ChatGPT launched in November 202...,"ChatGPT was launched in November 2022, marking...",single_hop_specifc_query_synthesizer
2,What are the most common Generalized Work Acti...,[Variation by Occupation Figure 23 presents va...,The data presents the frequency ranking of the...,single_hop_specifc_query_synthesizer
3,what SOC2 code 13 mean for jobs? i see it in t...,[Variation by Occupation Figure 23 presents va...,SOC2 code 13 is used for management and busine...,single_hop_specifc_query_synthesizer
4,How does the variation in ChatGPT usage by occ...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The variation in ChatGPT usage by occupation i...,multi_hop_abstract_query_synthesizer
5,how chatgpt get used different by jobs and how...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,"chatgpt use is different by occupation, with p...",multi_hop_abstract_query_synthesizer
6,How does the variation in ChatGPT usage by occ...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The variation in ChatGPT usage by occupation i...,multi_hop_abstract_query_synthesizer
7,"How do demographic trends such as age, gender,...",[<1-hop>\n\nVariation by Occupation Figure 23 ...,Demographic trends and occupational variation ...,multi_hop_abstract_query_synthesizer


# 2. Baseline RAG pipeline (Naive)

In [8]:
# Langchain RAG pipeline
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain.prompts import ChatPromptTemplate



# R - Retrieval

# Load
path = "data/"
loader = DirectoryLoader(path, glob = "*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

# Chunk
text_splitter= RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap=0)
split_documents = text_splitter.split_documents(docs)
print(len(split_documents))

# Embed
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# Qdrant Vector Store
client = QdrantClient(":memory:")
client.create_collection(
    collection_name = 'advanced_build_data',
    vectors_config = VectorParams(size=1536, distance= Distance.COSINE)
)
vector_store = QdrantVectorStore(
    client = client,
    collection_name = 'advanced_build_data',
    embedding = embeddings

)

# Add documents
_ = vector_store.add_documents(split_documents)

# Retriever
retriever = vector_store.as_retriever(search_kwargs = {'k': 3})

# Node retieve
def retrieve(state):
    retrieved_docs = retriever.invoke(state["question"])
    return {"context": retrieved_docs}


# A - Augmented

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""
rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

# G - Generation

# model for generation 
llm = ChatOpenAI(model = "gpt-4.1-nano")

# Node generate
def generate(state):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(messages)
    return {"response" : response.content}




275


In [9]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

# State
class State(TypedDict):
    question: str
    context: List[Document]
    response: str

In [10]:
# Graph
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
naive_graph= graph_builder.compile()

# 3. Improved RAG pipeline (semantic chunking)

We are going to  use semantic chunking strategy:
 
https://python.langchain.com/api_reference/experimental/text_splitter/langchain_experimental.text_splitter.SemanticChunker.html

Specifically we are going to implement semantic chunking strategy that:
- Groups semantically similar sentences together
- Uses a threshold to determine similarity
- Chunks paragraphs greedily up to max size
- Minimum chunk = single sentence


Semantic chunking groups sentences by **meaning**, so retrieved chunks are more coherent and relevant, improving retrieval precision and downstream faithfulness/helpfulness.
It reduces boundary breaks and context fragmentation versus fixed-size splits, often requiring fewer chunks to answer a query accurately.

In [ ]:
# Semantic chunking strategy

from langchain_experimental.text_splitter import SemanticChunker

semantic_splitter = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"), # measure semantic shifts between sentences -> same as retriever for alignment
    breakpoint_threshold_type="percentile", #decide where to cut: when distance is above a chosen percentile
    breakpoint_threshold_amount=90,      # 90–99: lower -> more cuts; higher -> fewer cuts
    sentence_split_regex=r"(?<=[.?!])\s+", #regex for sentence split
    min_chunk_size=None # no charcter floor
)

split_documents = semantic_splitter.split_documents(docs)
len(split_documents)

171

In [26]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="advanced_build_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client = client,
    collection_name = "advanced_build_data",
    embedding = embeddings
)

_ = vector_store.add_documents(documents=split_documents)

semantic_retriever = vector_store.as_retriever(search_kwargs = {'k': 3})

In [27]:
# Adjusted node for retrieval
def retrieved_semantic(state):
    retrieved_docs = semantic_retriever.invoke(state["question"])
    return {"context": retrieved_docs}

In [28]:
# Rebuild the graph with the new retriever

class SemanticState(TypedDict):
    question: str
    context: List[Document]
    response: str

semantic_graph_builder = StateGraph(SemanticState).add_sequence([retrieved_semantic, generate])
semantic_graph_builder.add_edge(START, "retrieved_semantic")
semantic_graph = semantic_graph_builder.compile()

# 4. RAGAS Evaluation

In [15]:
# judge model

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model = "gpt-4.1-mini"))

#### The metrics:

#### Faithfulness 

The Faithfulness metric measures how factually consistent a **response** is with the **retrieved context**. It ranges from 0 to 1, with higher scores indicating better consistency.

A response is considered faithful if all its claims can be supported by the retrieved context.


#### Response Relevancy

Response Relevancy, focuses on assessing how pertinent the generated answer is to the given prompt. A lower score is assigned to answers that are incomplete or contain redundant information and higher scores indicate better relevancy. 

**The metric turns your answer into one or more hypothetical questions and then compares those (via embeddings) to the original question. If the answer is on-topic, the questions it implies will be close to the user’s question.**

This metric is computed using the **user_input**, the **retrieved_context** and the **response**.

The Answer Relevancy is defined as the mean cosine similarity of the original question to a number of artifical questions, which where generated (reverse engineered) based on the answer.

*Why the context?* It is also shown to generate the answer.

#### Context Precision

Context Precision is a metric that evaluates the retriever’s ability to rank relevant chunks higher than irrelevant ones for a given query in the retrieved context. Specifically, it assesses the **degree to which relevant chunks in the retrieved context are placed at the top of the ranking**.

It is calculated as the mean of the precision@k for each chunk in the context. Precision@k is the ratio of the number of relevant chunks at rank k to the total number of chunks at rank k.

In RAGAS, it uses *LLMContextPrecisionWithReference*, that means, to estimate if the retrieved contexts are relevant, this method uses the LLM to compare each chunk in **retrieved_contexts** with the **reference**. 

You will be needing: **user_input**, **retrived_contexts**, **reference**.

#### Context Recall

Context Recall measures how many of the relevant documents (or pieces of information) were successfully retrieved. It focuses on not missing important results. Higher recall means fewer relevant documents were left out. In short, recall is about not missing anything important. Since it is about not missing anything, calculating context recall always requires a reference to compare against.

In new RAGAS we can use an LLM based approach: **LLMContextRecall**. It is computed using **user_input**, **reference** and the **retrieved_contexts**, and the values range between 0 and 1, with higher values indicating better performance. 
This metric uses reference as a proxy to reference_contexts which also makes it easier to use as annotating reference contexts can be very time-consuming. **To estimate context recall from the reference, the reference is broken down into claims each claim in the reference answer is analyzed to determine whether it can be attributed to the retrieved context or not**. In an ideal scenario, all claims in the reference answer should be attributable to the retrieved context.

#### Answer Correctness

The assessment of Answer Correctness involves gauging the accuracy of the **generated answer** when compared to the **ground truth**. This evaluation relies on the ground truth and the answer, with scores ranging from 0 to 1. A higher score indicates a closer alignment between the generated answer and the ground truth, signifying better correctness.

Answer correctness encompasses two critical aspects: semantic similarity between the generated answer and the ground truth, as well as factual similarity. These aspects are combined using a weighted scheme to formulate the answer correctness score. Users also have the option to employ a ‘threshold’ value to round the resulting score to binary, if desired.


#### Baseline system eval

In [16]:
# Running the synthetic questions through the baseline RAG pipeline
for test_row in dataset:
  response = naive_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [17]:
# Convert synthetic dataset to evaluation format
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

In [18]:
# Evaluate Baseline RAG system

from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall, AnswerCorrectness
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)


baseline_result = evaluate(
    dataset = evaluation_dataset,
    metrics = [Faithfulness(), ResponseRelevancy(), LLMContextPrecisionWithReference(), LLMContextRecall(), AnswerCorrectness()],
    llm = evaluator_llm,
    run_config = custom_run_config
)


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

In [19]:
# Check the results
baseline_result

{'faithfulness': 0.6603, 'answer_relevancy': 0.6977, 'llm_context_precision_with_reference': 0.8750, 'context_recall': 0.6750, 'answer_correctness': 0.6111}

#### Semantic system eval

In [29]:
# Running the synthetic questions through the semantic RAG pipeline
import copy

semantic_dataset = copy.deepcopy(dataset)

for test_row in semantic_dataset:
  response = semantic_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [30]:
# Convert synthetic dataset to evaluation format

semantic_evaluation_dataset = EvaluationDataset.from_pandas(semantic_dataset.to_pandas())

In [31]:
# Evaluate semantic RAG system
semantic_result = evaluate(
    dataset=semantic_evaluation_dataset,
    metrics=[Faithfulness(), ResponseRelevancy(), LLMContextPrecisionWithReference(), LLMContextRecall(), AnswerCorrectness() ],
    llm=evaluator_llm,
    run_config=custom_run_config
)




Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

In [32]:
semantic_result

{'faithfulness': 0.8919, 'answer_relevancy': 0.9482, 'llm_context_precision_with_reference': 0.8854, 'context_recall': 0.7250, 'answer_correctness': 0.5427}

# 5: Analysis


In [33]:
# Analysis: Baseline vs Semantic Chunking Results
import pandas as pd
import numpy as np

# Extract metrics from both results
baseline_metrics = baseline_result.to_pandas()
semantic_metrics = semantic_result.to_pandas()

# Find only numeric metric columns
metric_columns = []
for col in baseline_metrics.columns:
    if col not in ['question', 'answer', 'contexts', 'ground_truth', 'user_input', 'response', 'retrieved_contexts', 'reference_contexts']:
        # Check if column contains numeric data
        try:
            pd.to_numeric(baseline_metrics[col], errors='raise')
            metric_columns.append(col)
        except (ValueError, TypeError):
            continue

print(f"Found {len(metric_columns)} numeric metric columns: {metric_columns}")

# Create comparison DataFrame
comparison_data = {
    'Metric': metric_columns,
    'Baseline': [round(baseline_metrics[col].mean(), 4) for col in metric_columns],
    'Semantic': [round(semantic_metrics[col].mean(), 4) for col in metric_columns]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df['Delta'] = comparison_df['Semantic'] - comparison_df['Baseline']

print("\n=== METRIC COMPARISON ===")
print(comparison_df.to_string(index=False))


Found 5 numeric metric columns: ['faithfulness', 'answer_relevancy', 'llm_context_precision_with_reference', 'context_recall', 'answer_correctness']

=== METRIC COMPARISON ===
                              Metric  Baseline  Semantic   Delta
                        faithfulness    0.6603    0.8919  0.2316
                    answer_relevancy    0.6977    0.9482  0.2505
llm_context_precision_with_reference    0.8750    0.8854  0.0104
                      context_recall    0.6750    0.7250  0.0500
                  answer_correctness    0.6111    0.5427 -0.0684


Semantic chunking made retrieval more on-topic and grounded, boosting faithfulness and answer relevancy (plus slight gains in context precision/recall).
There’s a small answer correctness dip, suggesting we should fine-tune chunk size/reranking to keep precision while preserving the grounding gains.